In [1]:
import os
import json 
import time 
import re
import random
import numpy as np 
from tqdm.auto import tqdm
from util.utils import set_seed, read_data, save_result, get_answer_from_text, chat_huggingface
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, AutoModel
os.environ["CUDA_VISIBLE_DEVICES"] = "6"

seed = 42
set_seed(seed)

/drive2/ryusejong/miniconda3/envs/llm1/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# LFF v3

## Embedding Cosine similarity

In [2]:
base_output_path = "./output/GSM8K_Llama-3-8B-Instruct_zeroshot_CoT_test_512.jsonl"
lff3_output_path = "./output/GSM8K_Llama-3-8B-Instruct_LFF_v3_test_512.jsonl"
lff3_similarity_path = "./output/GSM8K_Llama-3-8B-Instruct_LFF_v3_test_512_sim.jsonl"

base_output = read_data(base_output_path)
lff3_output = read_data(lff3_output_path)
lff3_similarity = read_data(lff3_similarity_path)

print(f"base_output: {len(base_output)}")
print(f"lff3_output: {len(lff3_output)}")
print(f"lff3_similarity: {len(lff3_similarity)}")

base_output: 1319
lff3_output: 1319
lff3_similarity: 1319


In [3]:
all_similarity = []
correct_similarity = []
incorrect_similarity = []

for i in tqdm(range(len(base_output))):
    true_answer = base_output[i]["answer"]
    pred_answer = base_output[i]["pred_ans"]
    similarity = lff3_similarity[i]["max_sim"]

    all_similarity.append(similarity)

    if true_answer == pred_answer:
        correct_similarity.append(similarity)
    else:
        incorrect_similarity.append(similarity)

print(f"all_similarity: {len(all_similarity)}")       
print(f"all_similarity: {np.mean(all_similarity)}")
print(f"correct_similarity: {len(correct_similarity)}")
print(f"correct_similarity: {np.mean(correct_similarity)}")
print(f"incorrect_similarity: {len(incorrect_similarity)}")
print(f"incorrect_similarity: {np.mean(incorrect_similarity)}")

100%|██████████| 1319/1319 [00:00<00:00, 866179.27it/s]

all_similarity: 1319
all_similarity: 0.28530950886087947
correct_similarity: 1074
correct_similarity: 0.28463978119180633
incorrect_similarity: 245
incorrect_similarity: 0.2882453762755102


## Correct - Incorrect

In [5]:
correct_correct = []
correct_incorrect = []
incorrect_correct = []
incorrect_incorrect = []

for i in tqdm(range(len(base_output))):
    true_answer = base_output[i]["answer"]
    base_pred_answer = base_output[i]["pred_ans"]
    lff3_pred_answer = lff3_output[i]["pred_ans2"]

    if base_pred_answer == true_answer:
        if lff3_pred_answer == true_answer:
            correct_correct.append(lff3_output[i])
        else:
            correct_incorrect.append(lff3_output[i])
    else:
        if lff3_pred_answer == true_answer:
            incorrect_correct.append(lff3_output[i])
        else:
            incorrect_incorrect.append(lff3_output[i])

print(f"correct_correct: {len(correct_correct)}\nIndex: {[o['index'] for o in correct_correct]}\n")
print(f"correct_incorrect: {len(correct_incorrect)}\nIndex: {[o['index'] for o in correct_incorrect]}\n")
print(f"incorrect_correct: {len(incorrect_correct)}\nIndex: {[o['index'] for o in incorrect_correct]}\n")
print(f"incorrect_incorrect: {len(incorrect_incorrect)}\nIndex: {[o['index'] for o in incorrect_incorrect]}\n")
print(f"total num: {len(correct_correct) + len(correct_incorrect) + len(incorrect_correct) + len(incorrect_incorrect)}")

100%|██████████| 1319/1319 [00:00<00:00, 744687.98it/s]

correct_correct: 658
Index: [0, 1, 3, 5, 9, 10, 11, 15, 16, 18, 23, 25, 27, 28, 31, 35, 42, 44, 45, 48, 49, 52, 54, 55, 56, 58, 59, 65, 67, 68, 69, 71, 72, 77, 78, 79, 80, 81, 82, 84, 85, 86, 88, 89, 90, 92, 93, 95, 96, 98, 105, 106, 109, 110, 113, 114, 115, 117, 120, 121, 122, 123, 126, 129, 130, 131, 132, 133, 134, 137, 139, 140, 144, 146, 149, 151, 152, 154, 155, 158, 159, 163, 164, 165, 166, 169, 170, 171, 173, 176, 178, 179, 181, 182, 185, 190, 193, 194, 200, 202, 204, 206, 208, 212, 213, 215, 217, 219, 222, 223, 225, 227, 228, 229, 230, 231, 232, 235, 237, 238, 239, 240, 242, 243, 244, 247, 252, 255, 256, 258, 261, 262, 266, 269, 271, 273, 274, 275, 276, 277, 278, 281, 282, 285, 286, 287, 289, 290, 291, 292, 294, 296, 299, 304, 305, 306, 308, 312, 313, 319, 321, 322, 323, 326, 328, 329, 331, 332, 334, 336, 337, 338, 339, 342, 343, 344, 346, 347, 348, 350, 351, 355, 357, 360, 365, 369, 372, 374, 375, 376, 377, 379, 381, 384, 385, 386, 387, 388, 389, 390, 391, 392, 393, 395, 397, 4